In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
df = pd.read_csv('../Data Files/Raw Files Prototype/credit_card_frauds.csv')
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace(r"[()\-\/]", "", regex=True)
)

In [3]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

df['hour'] = df['trans_date_trans_time'].dt.hour
df['day_of_week']  = df['trans_date_trans_time'].dt.dayofweek
df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
df['is_late_night'] = df['hour'].between(1, 4).astype(int)

In [4]:
df['log_amt'] = np.log1p(df['amt'])

df['amt_zscore'] = (df['amt'] - df['amt'].mean()) / df['amt'].std()

df['is_high_amt'] = (df['amt'] > df['amt'].quantile(0.95)).astype(int)

In [5]:
def haversine_distance(lat1,lon1,lat2,lon2):
  earth_radius = 6371
  lat1,lon1,lat2,lon2 = map(np.radians, [lat1,lon1,lat2,lon2])

  dlat = lat2 - lat1
  dlon = lon2 - lon1
    
  a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
  return earth_radius * 2 * np.arcsin(np.sqrt(a))

In [6]:
df['distance_from_home'] = haversine_distance(
    df['lat'],
    df['long'],
    df['merch_lat'],
    df['merch_long']
)

df['log_distance'] = np.log1p(df['distance_from_home'])

In [7]:
df['dob'] = pd.to_datetime(df['dob'],format='%Y-%m-%d')

today = pd.Timestamp('today')
df['customer_age'] = ((today - df['dob']).dt.days / 365.25).astype(int)

df['age_group'] = pd.cut(df['customer_age'], 
                          bins=[0, 25, 35, 50, 65, 100],
                          labels=['18-25', '26-35', '36-50', '51-65', '65+'])

df['age_group_encoded'] = df['age_group'].cat.codes

In [8]:
df['category'].value_counts()

category
gas_transport     35089
grocery_pos       32732
home              32516
shopping_pos      30329
kids_pets         29704
shopping_net      26379
personal_care     24406
entertainment     24222
food_dining       23038
health_fitness    22593
misc_pos          20024
misc_net          16898
grocery_net       11355
travel            10322
Name: count, dtype: int64

In [9]:
df['category_encoded'] = df.groupby('category')['is_fraud'].transform('mean')

In [10]:
df.select_dtypes(include='object').columns

Index(['merchant', 'category', 'city', 'state', 'job', 'trans_num'], dtype='object')

In [11]:
global_fraud_rate = df['is_fraud'].mean()
smoothing = 10

city_stats = df.groupby('city').agg(
    fraud_rate=('is_fraud', 'mean'),
    txn_count=('is_fraud', 'count')
).reset_index()

city_stats['city_encoded'] = (
    (city_stats['fraud_rate'] * city_stats['txn_count'] + global_fraud_rate * smoothing) /
    (city_stats['txn_count'] + smoothing)
)

city_map = city_stats.set_index('city')['city_encoded']
df['city_encoded'] = df['city'].map(city_map)
df['city_encoded'] = df['city_encoded'].fillna(global_fraud_rate)

# Same for state (simple target encoding — fewer unique values)
state_fraud_rate = df.groupby('state')['is_fraud'].mean()
df['state_encoded'] = df['state'].map(state_fraud_rate)
df['state_encoded'] = df['state_encoded'].fillna(global_fraud_rate)

print(df[['city_encoded', 'state_encoded']].describe())

        city_encoded  state_encoded
count  339607.000000  339607.000000
mean        0.005026       0.005247
std         0.012541       0.001417
min         0.000018       0.003972
25%         0.002518       0.004660
50%         0.004107       0.004994
75%         0.005470       0.005165
max         0.656982       0.016875


In [12]:
global_fraud_rate = df['is_fraud'].mean()
smoothing = 10

merchant_stats = df.groupby('merchant').agg(
    fraud_rate=('is_fraud', 'mean'),
    txn_count=('is_fraud', 'count')
).reset_index()

merchant_stats['merchant_encoded'] = (
    (merchant_stats['fraud_rate'] * merchant_stats['txn_count'] + global_fraud_rate * smoothing) /
    (merchant_stats['txn_count'] + smoothing)
)

merchant_map = merchant_stats.set_index('merchant')['merchant_encoded']
df['merchant_encoded'] = df['merchant'].map(merchant_map)
df['merchant_encoded'] = df['merchant_encoded'].fillna(global_fraud_rate)

In [13]:
job_stats = df.groupby('job').agg(
    fraud_rate=('is_fraud', 'mean'),
    txn_count=('is_fraud', 'count')
).reset_index()

job_stats['job_encoded'] = (
    (job_stats['fraud_rate'] * job_stats['txn_count'] + global_fraud_rate * smoothing) /
    (job_stats['txn_count'] + smoothing)
)

job_map = job_stats.set_index('job')['job_encoded']
df['job_encoded'] = df['job'].map(job_map)
df['job_encoded'] = df['job_encoded'].fillna(global_fraud_rate)

# Confirm
print(df[['merchant_encoded', 'job_encoded']].describe())

       merchant_encoded    job_encoded
count     339607.000000  339607.000000
mean           0.005251       0.005033
std            0.005628       0.011567
min            0.000075       0.000024
25%            0.001523       0.002745
50%            0.003310       0.004104
75%            0.007598       0.005738
max            0.033997       0.656982


In [14]:
columsn_to_drop = ['age_group', 'trans_date_trans_time', 'dob', 'category', 'city', 'state', 'trans_num', 'job', 'merchant']
df.drop(columns = columsn_to_drop, inplace=True)

In [15]:
df.head()

,amt,lat,long,city_pop,merch_lat,merch_long,is_fraud,hour,day_of_week,is_weekend,...,is_high_amt,distance_from_home,log_distance,customer_age,age_group_encoded,category_encoded,city_encoded,state_encoded,merchant_encoded,job_encoded
0,107.23,48.8878,-118.2105,149,49.159047,-118.186462,0,0,1,0,...,0,30.212176,3.440808,47,2,0.013229,0.001613,0.004660,0.007270,0.001613
1,220.11,42.1808,-112.2620,4154,43.150704,-112.154481,0,0,1,0,...,1,108.206083,4.693237,64,3,0.002271,0.010809,0.004107,0.002139,0.021318
2,96.29,41.6125,-122.5258,589,41.657520,-122.230347,0,0,1,0,...,0,25.059079,3.260366,80,4,0.013229,0.005911,0.004994,0.013716,0.006364
3,7.77,32.9396,-105.8189,899,32.863258,-106.520205,0,0,1,0,...,0,66.021685,4.205016,58,3,0.006166,0.003770,0.005165,0.009906,0.006548
4,6.85,43.0172,-111.0292,471,43.753735,-111.454923,0,0,1,0,...,0,88.830984,4.497930,58,3,0.003096,0.005125,0.004284,0.002494,0.005125


In [16]:
df.columns

Index(['amt', 'lat', 'long', 'city_pop', 'merch_lat', 'merch_long', 'is_fraud',
       'hour', 'day_of_week', 'is_weekend', 'is_late_night', 'log_amt',
       'amt_zscore', 'is_high_amt', 'distance_from_home', 'log_distance',
       'customer_age', 'age_group_encoded', 'category_encoded', 'city_encoded',
       'state_encoded', 'merchant_encoded', 'job_encoded'],
      dtype='object')

In [17]:
df.to_csv('../Data Files/Processed Files/credit_card_fraud_clean.csv', index = False)